# The Last Call Is Not the Best Call: A Bank Campaign Efficiency Audit

**Decision question:** How can a bank increase term-deposit conversions without simply calling customers more often—and which information is genuinely available before a call?

This analysis examines the supplied `bank.csv`: 11,162 campaign records and 17 columns. The target is `deposit` (yes/no). The file has no missing cells or exact duplicate rows, but `pdays = -1` and category value `unknown` have meaning and must not be treated as ordinary numeric/category values without thought.

**Distinctive angle:** separate *pre-call targeting* from *after-call explanation*. Call duration is known only after a call, so using it to decide whom to call creates target leakage for a campaign-planning model. We quantify this difference, study repeat-contact saturation and outcome differences by segment, and build a transparent contact-priority score evaluated on a holdout.

**Scope caution:** No campaign date/year, customer ID, deposit amount, contact cost, or profit is included. We can estimate conversion efficiency and relative lift, not ROI, incremental causal impact, or customer lifetime value.

In [ ]:
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve

sns.set_theme(style='whitegrid'); plt.rcParams['figure.figsize']=(10,5.5)
DATA_PATH=Path('bank.csv')  # place beside notebook, or change to Path('data/bank.csv')
OUT=Path('images_bank'); OUT.mkdir(exist_ok=True)
raw=pd.read_csv(DATA_PATH)
print('Shape:',raw.shape,'| exact duplicates:',raw.duplicated().sum(),'| missing cells:',raw.isna().sum().sum())
raw.head()

## 1. Data validation and feature preparation
The file uses `pdays = -1` to mean the client was not previously contacted. We preserve that meaning by creating `previously_contacted` and set `pdays` to missing for such cases. `unknown` categories remain explicit. Campaign count is treated as contact pressure. Duration is reserved for retrospective analysis and excluded from pre-call predictors.

In [ ]:
required={'age','job','marital','education','default','balance','housing','loan','contact','day','month','duration','campaign','pdays','previous','poutcome','deposit'}
assert required.issubset(raw.columns), f'Missing columns: {required-set(raw.columns)}'
assert set(raw.deposit.dropna().unique()) <= {'yes','no'}
df=raw.drop_duplicates().copy()
df['deposit_yes']=(df.deposit=='yes').astype(int)
df['previously_contacted']=(df.pdays!=-1).astype(int)
df.loc[df.pdays==-1,'pdays']=np.nan
df['campaign_band']=pd.cut(df.campaign,[0,1,2,3,5,10,np.inf],labels=['1','2','3','4-5','6-10','11+'])
df['age_band']=pd.cut(df.age,[17,24,34,44,54,64,100],labels=['18-24','25-34','35-44','45-54','55-64','65+'])
df['balance_band']=pd.cut(df.balance,[-np.inf,0,500,2000,10000,np.inf],labels=['negative','0-499','500-1,999','2,000-9,999','10,000+'])
print('Rows:',len(df),'| conversion:',f"{df.deposit_yes.mean():.1%}")
print('Target counts:'); display(df.deposit.value_counts().rename_axis('deposit').to_frame('records'))
print('pdays = -1 records:',(raw.pdays==-1).sum())

## 2. Campaign baseline and conversion segments
First establish the overall conversion baseline, then compare rates with uncertainty intervals. Segment results are descriptive: the file does not show random assignment, so differences do not prove a campaign treatment caused the outcome.

In [ ]:
def wilson(k,n,z=1.96):
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n); a=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n)); return (c-a)/d,(c+a)/d
def segment(col):
    x=df.groupby(col,observed=True).deposit_yes.agg(['size','sum']).rename(columns={'size':'n','sum':'yes'})
    x['rate']=x.yes/x.n; ci=[wilson(k,n) for k,n in zip(x.yes,x.n)]
    x['lo']=[v[0] for v in ci]; x['hi']=[v[1] for v in ci]; x['lift_vs_avg']=x.rate/df.deposit_yes.mean()-1
    return x
print('Overall subscription rate:',f"{df.deposit_yes.mean():.2%}")
fig,axes=plt.subplots(2,3,figsize=(16,10))
for ax,col in zip(axes.flat,['job','education','marital','housing','loan','contact']):
    x=segment(col); xx=np.arange(len(x))
    ax.bar(xx,x.rate,color='#2878b5',yerr=[x.rate-x.lo,x.hi-x.rate],capsize=3)
    ax.axhline(df.deposit_yes.mean(),color='#c53030',ls='--',lw=1)
    ax.set_xticks(xx); ax.set_xticklabels(x.index,rotation=35,ha='right',fontsize=8); ax.set_title(col); ax.set_ylabel('Subscription rate')
fig.suptitle('Subscription rate by customer/contact segment (95% Wilson intervals)',y=1.02)
plt.tight_layout(); plt.savefig(OUT/'01_conversion_segments.png',dpi=160); plt.show()
display(segment('job').sort_values('rate',ascending=False).style.format({'rate':'{:.1%}','lo':'{:.1%}','hi':'{:.1%}','lift_vs_avg':'{:+.1%}'}))

## 3. Contact pressure: when do additional attempts stop helping?
Compare conversion by contacts made in the current campaign. High campaign counts may be associated with harder-to-convert customers, so this is not a causal estimate of the effect of making another call. Sparse counts are pooled into bands to reduce noisy conclusions.

In [ ]:
x=segment('campaign_band').dropna()
display(x[['n','yes','rate','lo','hi','lift_vs_avg']].style.format({'rate':'{:.1%}','lo':'{:.1%}','hi':'{:.1%}','lift_vs_avg':'{:+.1%}'}))
fig,ax=plt.subplots(); xx=np.arange(len(x))
ax.errorbar(xx,x.rate,yerr=[x.rate-x.lo,x.hi-x.rate],fmt='o-',capsize=4,color='#2b6cb0')
ax.set_xticks(xx); ax.set_xticklabels(x.index); ax.axhline(df.deposit_yes.mean(),color='#c53030',ls='--')
ax.set(title='Subscription rate by number of current-campaign contacts',xlabel='Contact count band',ylabel='Subscription rate')
plt.tight_layout(); plt.savefig(OUT/'02_contact_saturation.png',dpi=160); plt.show()

## 4. Calendar pattern and prior campaign outcomes
This dataset provides month and day-of-month but no year; therefore this is a within-file calendar comparison, not a trend over time. Prior campaign outcome may be unknown for many observations, which is itself reported rather than discarded.

In [ ]:
month_order=['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
mx=segment('month').reindex(month_order)
fig,ax=plt.subplots(figsize=(11,5)); ax.bar(mx.index,mx.rate,color='#38a169')
ax.errorbar(np.arange(len(mx)),mx.rate,yerr=[mx.rate-mx.lo,mx.hi-mx.rate],fmt='none',ecolor='black',capsize=2)
ax.axhline(df.deposit_yes.mean(),color='#c53030',ls='--'); ax.set(title='Subscription rate by month (year not provided)',ylabel='Subscription rate')
plt.tight_layout(); plt.savefig(OUT/'03_monthly_conversion.png',dpi=160); plt.show()
display(segment('poutcome').sort_values('rate',ascending=False).style.format({'rate':'{:.1%}','lo':'{:.1%}','hi':'{:.1%}','lift_vs_avg':'{:+.1%}'}))

## 5. Statistical evidence and practical significance
Chi-square tests measure whether categorical segment and outcome are associated; Cramer's V gives effect size. Large samples can make small differences statistically significant. We report both, and never interpret association as causal impact.

In [ ]:
catcols=['job','education','marital','housing','loan','contact','month','campaign_band','poutcome']
rows=[]
for c in catcols:
    tab=pd.crosstab(df[c],df.deposit_yes); chi,p,_,_=stats.chi2_contingency(tab)
    v=np.sqrt(chi/(tab.values.sum()*(min(tab.shape)-1)))
    rows.append({'variable':c,'chi2':chi,'p_value':p,'cramers_v':v})
tests=pd.DataFrame(rows).sort_values('cramers_v',ascending=False); display(tests.style.format({'chi2':'{:.1f}','p_value':'{:.2e}','cramers_v':'{:.3f}'}))

## 6. Pre-call targeting model versus post-call leakage
We compare two prediction settings on the same stratified 25% holdout. **Pre-call model** uses only fields known before dialing (demographics, balance, existing loans, contact history, month/day, campaign number). It excludes `duration`. **Post-call diagnostic model** adds call duration, which is not available when deciding whom to call; its score is not a valid pre-call targeting performance estimate.

Logistic regression is the transparent baseline and histogram gradient boosting captures non-linear patterns. ROC-AUC ranks outcomes; average precision is useful with class imbalance; Brier score assesses probability accuracy. Calibration is shown for the pre-call boosting model.

In [ ]:
pre_num=['age','balance','day','campaign','previous','previously_contacted']
pre_cat=['job','marital','education','default','housing','loan','contact','month','poutcome']
def makepipe(num,cat,model):
    prep=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),
                           ('cat',OneHotEncoder(handle_unknown='ignore'),cat)])
    return Pipeline([('prep',prep),('model',model)])
tr,te=train_test_split(df,test_size=.25,stratify=df.deposit_yes,random_state=42)
specs={('Pre-call','Logistic'):(pre_num,pre_cat,LogisticRegression(max_iter=2000)),
       ('Pre-call','Boosting'):(pre_num,pre_cat,HistGradientBoostingClassifier(random_state=42)),
       ('Post-call diagnostic','Boosting'):(pre_num+['duration'],pre_cat,HistGradientBoostingClassifier(random_state=42))}
pred={}; rows=[]
for key,(num,cat,model) in specs.items():
    cols=num+cat; pipe=makepipe(num,cat,model).fit(tr[cols],tr.deposit_yes); p=pipe.predict_proba(te[cols])[:,1]; pred[key]=p
    rows.append({'setting':key[0],'model':key[1],'ROC-AUC':roc_auc_score(te.deposit_yes,p),'Avg precision':average_precision_score(te.deposit_yes,p),'Brier':brier_score_loss(te.deposit_yes,p)})
model_scores=pd.DataFrame(rows); display(model_scores.style.format({'ROC-AUC':'{:.3f}','Avg precision':'{:.3f}','Brier':'{:.3f}'}))
f,o=calibration_curve(te.deposit_yes,pred[('Pre-call','Boosting')],n_bins=10,strategy='quantile')
fig,ax=plt.subplots(); ax.plot([0,1],[0,1],'k--',label='Perfect calibration'); ax.plot(o,f,'o-',label='Pre-call boosting')
ax.set(xlabel='Mean predicted conversion probability',ylabel='Observed conversion rate',title='Pre-call model calibration on holdout'); ax.legend()
plt.tight_layout(); plt.savefig(OUT/'04_pre_call_calibration.png',dpi=160); plt.show()
print('Post-call AUC uplift is not available at decision time; do not use it as a targeting claim.')

## 7. Contact-priority simulation: lift, not invented revenue
Rank the held-out customers by the **pre-call** model's predicted probability. For a fixed top fraction of calls, compare conversion with the full holdout's random-selection baseline. This estimates predictive concentration in this sample, not incremental causal lift; no contact-cost or deposit-value data exist. The model and threshold are not tuned on the holdout labels.

In [ ]:
scored=te[['deposit_yes']].copy(); scored['score']=pred[('Pre-call','Boosting')]
base=scored.deposit_yes.mean(); fractions=np.linspace(.05,1,20); curve=[]
for frac in fractions:
    top=scored.nlargest(max(1,int(len(scored)*frac)),'score')
    curve.append({'share_called':frac,'conversion_top':top.deposit_yes.mean(),'lift_vs_baseline':top.deposit_yes.mean()/base,'share_of_all_yes_captured':top.deposit_yes.sum()/scored.deposit_yes.sum()})
lift=pd.DataFrame(curve)
fig,ax=plt.subplots(); ax.plot(lift.share_called*100,lift.conversion_top*100,marker='o',label='Model-ranked top share')
ax.axhline(base*100,color='#c53030',ls='--',label='Random-selection baseline')
ax.set(xlabel='Share of customers called (%)',ylabel='Conversion rate among called (%)',title='Pre-call targeting: conversion concentration on holdout'); ax.legend()
plt.tight_layout(); plt.savefig(OUT/'05_targeting_lift.png',dpi=160); plt.show()
display(lift.iloc[[0,4,9,14,19]].style.format({'share_called':'{:.0%}','conversion_top':'{:.1%}','lift_vs_baseline':'{:.2f}x','share_of_all_yes_captured':'{:.1%}'}))

## 8. Decision memo (complete after running)
1. **Campaign efficiency:** state the overall subscription rate and the contact-count point where observed conversion is lowest/highest; explain that this is association, not a causal call effect.
2. **Targeting:** report pre-call holdout AUC, average precision, and top-10% conversion lift versus baseline.
3. **Leakage lesson:** quantify post-call model performance but explicitly state duration is unavailable before dialing.
4. **Action:** recommend a controlled randomized pilot among score bands before reallocating the bank's actual call budget. Track cost per call, deposit amount, margin, complaints and opt-outs.
5. **Caveat:** no causal lift or ROI can be computed from this file alone.

## 9. Limitations and responsible use
- The dataset includes no campaign year, customer identifier, deposit amount, revenue, call cost, margin, consent, complaint or opt-out outcome. Thus no ROI, lifetime value, customer-level leakage check, or causal incremental lift is possible.
- `duration` occurs after dialing and is leakage for pre-call targeting; it is shown only as a diagnostic.
- Campaign contact counts are observational; repeat contacts may be targeted at harder-to-convert customers.
- No randomized treatment assignment is supplied. Segment gaps and model scores do not prove a call caused subscription.
- Calendar months have no year, so no year-over-year trend can be calculated.
- Predictive models can encode historical inequities. Review fairness, privacy, consent, and local rules before operational use.
- Model results are from one split; validate temporally on later campaign waves if dates become available.

**Reproduction:** keep `bank.csv` beside this notebook; install `pandas`, `numpy`, `scipy`, `scikit-learn`, `matplotlib`, and `seaborn`; restart the kernel and run all cells. Figures export to `images_bank/`.